# Silver Layer: Customer Info CDC

**Source**: `workspace.bronze.crm_cust_info_cdc` (Bronze CDC table)  
**Target**: `workspace.silver.crm_customers_cdc`  
**Primary Key**: `cst_id` → `customer_id`

**CDC Process:**
1. Read from Bronze CDC table
2. **Watermark**: Filter for NEW records only (ingestion_timestamp > last processed)
3. Apply transformations (trim, normalize, rename)
4. **MERGE** into Silver (update existing, insert new)

**Key Difference from Regular Silver:**
- Regular: Read full Bronze, OVERWRITE Silver
- CDC: Read NEW Bronze records, MERGE into Silver

## Configuration

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType
from pyspark.sql.functions import trim, col, current_timestamp
from delta.tables import DeltaTable

# Table names
bronze_table = "workspace.bronze.crm_cust_info_cdc"
silver_table = "workspace.silver.crm_customers_cdc"

print("✓ Configuration loaded")
print(f"  Source (Bronze CDC): {bronze_table}")
print(f"  Target (Silver CDC): {silver_table}")

## Step 1: Determine Watermark

Get the maximum `ingestion_timestamp` from Silver table.  
This tells us which Bronze records we've already processed.

In [0]:
# Check if Silver table exists
silver_exists = spark.catalog.tableExists(silver_table)

if silver_exists:
    # Get max ingestion_timestamp from Silver (last processed timestamp)
    watermark = spark.sql(f"""
        SELECT COALESCE(MAX(ingestion_timestamp), '1900-01-01') as max_timestamp
        FROM {silver_table}
    """).collect()[0]["max_timestamp"]
    print(f"✓ Silver table exists")
    print(f"  Watermark (last processed): {watermark}")
else:
    # No Silver table yet - this is the first run
    watermark = "1900-01-01"
    print(f"ℹ️  Silver table doesn't exist - initial load")
    print(f"  Watermark: {watermark} (will process all Bronze records)")

## Step 2: Read NEW Records from Bronze CDC

Filter Bronze CDC table for records with `ingestion_timestamp > watermark`.  
This gives us only the NEW/CHANGED records since last run.

In [0]:
# Read only NEW records from Bronze CDC
df_bronze_new = spark.table(bronze_table).filter(
    col("ingestion_timestamp") > watermark
)

new_record_count = df_bronze_new.count()
print(f"📊 New records from Bronze: {new_record_count:,}")

if new_record_count == 0:
    print("\n⏭️  No new records to process - exiting")
    dbutils.notebook.exit("No new data")

# Show sample
print("\nSample of new records:")
df_bronze_new.select("cst_id", "cst_firstname", "cst_lastname", "ingestion_timestamp").limit(5).display()

## Step 3: Apply Silver Transformations

Same transformations as regular Silver:
1. Trim whitespace
2. Normalize marital_status and gender
3. Filter NULL customer_ids
4. Rename columns

In [0]:
# Start with new Bronze records
df = df_bronze_new

# 1. Trimming string columns
for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name, trim(col(field.name)))

print("✓ Trimmed string columns")

In [0]:
# 2. Normalize marital_status and gender
df = (
    df
    .withColumn(
        "cst_marital_status",
        F.when(F.upper(col("cst_marital_status")) == "S", "Single")
        .when(F.upper(col("cst_marital_status")) == "M", "Married")
        .otherwise("n/a")
    )
    .withColumn(
        "cst_gndr",
        F.when(F.upper(col("cst_gndr")) == "F", "Female")
        .when(F.upper(col("cst_gndr")) == "M", "Male")
        .otherwise("n/a")
    )
)

print("✓ Normalized marital_status and gender")

In [0]:
# 3. Remove records with NULL customer ID
df = df.filter(col("cst_id").isNotNull())

print(f"✓ Filtered NULL customer_ids")
print(f"  Valid records for Silver: {df.count():,}")

In [0]:
# 4. Rename columns to business-friendly names
rename_map = {
    "cst_id": "customer_id",
    "cst_key": "customer_number",
    "cst_firstname": "first_name",
    "cst_lastname": "last_name",
    "cst_marital_status": "marital_status",
    "cst_gndr": "gender",
    "cst_create_date": "created_date"
}

for old_name, new_name in rename_map.items():
    df = df.withColumnRenamed(old_name, new_name)

print("✓ Renamed columns to business-friendly names")

# Show transformed sample
print("\nTransformed sample:")
df.limit(5).display()

## Step 4: MERGE into Silver Table

**MERGE** (not overwrite!) ensures idempotent processing:
- **Matched records**: Update with latest data
- **New records**: Insert

This prevents duplicates and handles late-arriving updates.

In [0]:
if not silver_exists:
    # First run: Create Silver table
    print("Creating Silver table with initial data...")
    (df.write
      .format("delta")
      .mode("overwrite")
      .option("overwriteSchema", "true")
      .saveAsTable(silver_table)
    )
    print(f"\n✓ Silver table created: {spark.table(silver_table).count():,} records")
else:
    # Subsequent runs: MERGE new/updated records
    print("Merging new records into Silver table...")
    
    target_table = DeltaTable.forName(spark, silver_table)
    
    (target_table.alias("target")
      .merge(
          df.alias("source"),
          "target.customer_id = source.customer_id"  # Match on primary key
      )
      .whenMatchedUpdateAll()  # Update existing records
      .whenNotMatchedInsertAll()  # Insert new records
      .execute()
    )
    
    print(f"\n✓ MERGE complete")
    print(f"  Total records in Silver: {spark.table(silver_table).count():,}")

## Verification

In [0]:
%sql
-- Check Silver CDC table
SELECT 
  COUNT(*) as total_records,
  COUNT(DISTINCT customer_id) as unique_customers,
  MIN(ingestion_timestamp) as first_ingestion,
  MAX(ingestion_timestamp) as last_ingestion
FROM silver.crm_customers_cdc

In [0]:
%sql
-- Sample latest records
SELECT *
FROM silver.crm_customers_cdc
ORDER BY ingestion_timestamp DESC
LIMIT 10

In [0]:
%sql
-- Check normalized values
SELECT marital_status, gender, COUNT(*) as count
FROM silver.crm_customers_cdc
GROUP BY marital_status, gender
ORDER BY marital_status, gender